# Session 2: the equations, and two kinds of friction
**A. Real-world problem.** A warehouse needs a ramp so that crates slide down by themselves
and arrive at a gentle, known speed. Too shallow and the crate sticks; too steep and it crashes
into the stop. Which angle, for which surface?

**Plan.** (1) write the model with two friction coefficients, (2) find the acceleration,
(3) plot friction and acceleration against angle, (4) solve the design problem backwards.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

G = 9.81  # gravity, m/s^2

## Step 1: two friction coefficients
A block at rest is held by *static* friction, which can reach at most mu_s * N. Once it slides,
*kinetic* friction acts with a fixed size mu_k * N, usually a little smaller (mu_k <= mu_s).

    N = m g cos t        slope pull = m g sin t
    static: friction = slope pull   while   m g sin t <= mu_s m g cos t   i.e.  tan t <= mu_s
    sliding: m a = m g sin t - mu_k m g cos t   so   a = g (sin t - mu_k cos t)

Mass m cancels from a, and from the test tan t <= mu_s. That is why a heavy and a light
crate of the same material start sliding at the same angle.

In [ ]:
def normal_force(m, theta_deg, g=G):
    return m * g * np.cos(np.radians(theta_deg))


def friction_acting(m, theta_deg, mu_s, mu_k, g=G):
    """Friction force on a block released from rest (newtons): static while it holds, kinetic once it slips."""
    t = np.radians(theta_deg)
    if np.tan(t) <= mu_s:
        return m * g * np.sin(t)          # static friction equals the pull
    return mu_k * m * g * np.cos(t)       # kinetic friction


def accel_from_rest(theta_deg, mu_s, mu_k, g=G):
    """Acceleration down the slope (m/s^2) of a block released from rest. mu_k must not exceed mu_s."""
    t = np.radians(theta_deg)
    if np.tan(t) <= mu_s:
        return 0.0
    return float(g * (np.sin(t) - mu_k * np.cos(t)))


def slide_angle(mu_s):
    """Angle (degrees) where a block at rest first slips: tan t = mu_s."""
    return float(np.degrees(np.arctan(mu_s)))


def mu_s_from_angle(theta_deg):
    """Measure mu_s from the tilt angle at which slipping starts."""
    return float(np.tan(np.radians(theta_deg)))

## Step 2: from acceleration to speed and distance
Starting from rest with constant a: v = a t and distance x = (1/2) a t^2.

In [ ]:
def speed_after(a, t):
    return a * t


def distance_after(a, t):
    return 0.5 * a * t * t

## Step 3: the design problem, solved numerically
We want a given acceleration a_target for a known mu_k. a(t) rises steadily with angle, so
bisection (halve the search interval again and again) finds the angle. Afterwards we also
check that the angle is steep enough for the crate to start at all (tan t > mu_s).

In [ ]:
def angle_for_acceleration(a_target, mu_k, g=G, lo=0.0, hi=89.0, iters=60):
    """Angle (degrees) that gives sliding acceleration a_target for kinetic friction mu_k."""
    f = lambda d: g * (np.sin(np.radians(d)) - mu_k * np.cos(np.radians(d))) - a_target
    if f(lo) > 0 or f(hi) < 0:
        raise ValueError("target acceleration is outside what any angle can give")
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if f(mid) < 0:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

## Step 4: see it
Left: friction acting on a 10 kg crate against angle. It climbs with the pull, reaches mu_s N,
then *drops* to mu_k N when slipping starts. Right: the acceleration, zero until the slide
angle and then jumping up.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m, mu_s, mu_k = 10.0, 0.40, 0.30
    ang = np.linspace(0, 50, 501)
    fr = [friction_acting(m, d, mu_s, mu_k) for d in ang]
    ac = [accel_from_rest(d, mu_s, mu_k) for d in ang]
    print(f"slide angle for mu_s = {mu_s}: {slide_angle(mu_s):.2f} degrees")
    a30 = accel_from_rest(30, mu_s, mu_k)
    print(f"m = {m} kg, 30 degrees, mu_s = {mu_s}, mu_k = {mu_k}: a = {a30:.3f} m/s^2, "
          f"v(2 s) = {speed_after(a30, 2):.2f} m/s, distance(2 s) = {distance_after(a30, 2):.2f} m")
    a_t = 2 * 3.0 / 2.0 ** 2
    th = angle_for_acceleration(a_t, mu_k)
    print(f"design: 3 m in 2 s needs a = {a_t} m/s^2 -> angle {th:.2f} degrees "
          f"(starts since tan t = {np.tan(np.radians(th)):.3f} > {mu_s})")
    for d in (24, 25, 26):
        print(f"  integer angle {d}: a = {accel_from_rest(d, mu_s, mu_k):.3f} m/s^2, "
              f"distance in 2 s = {distance_after(accel_from_rest(d, mu_s, mu_k), 2):.2f} m")

    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(ang, fr, label="friction acting (N)")
    ax.plot(ang, [mu_s * normal_force(m, d) for d in ang], ls=":", color="gray", label="static limit mu_s N")
    ax.axvline(slide_angle(mu_s), color="red", ls="--", label="slide angle")
    ax.set_xlabel("ramp angle (degrees)")
    ax.set_ylabel("friction force (N)")
    ax.set_title("Friction rises, then drops when slipping starts (10 kg)")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "friction_vs_angle.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    ax2.plot(ang, ac, label="mu_s = 0.40, mu_k = 0.30")
    ax2.plot(ang, [accel_from_rest(d, 0.40, 0.40) for d in ang], ls="--", label="mu_s = mu_k = 0.40")
    ax2.axhline(a_t, color="green", ls=":", label="design target 1.5 m/s^2")
    ax2.plot([th], [a_t], "o", color="green")
    ax2.set_xlabel("ramp angle (degrees)")
    ax2.set_ylabel("acceleration from rest (m/s^2)")
    ax2.set_title("Acceleration jumps from 0 once the block slips")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "accel_vs_angle.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print `accel_from_rest` for angles 20 to 40 in steps of 5 with mu_s = 0.4 and mu_k = 0.3 and mark where the block starts to move.
2. **Stretch.** Use `angle_for_acceleration` to find the angle for a = 2.0 m/s^2 with mu_k = 0.3, and check it with the closed formula in Section B.
3. **Explain.** Explain why the same ramp can make a crate on a wet surface (smaller mu_s) start while the dry surface holds it.

## Assessment
- **A. Conceptual.** Why does a block often 'jerk' into motion rather than speeding up smoothly from rest? What happens to the friction force at the moment of slipping?
- **B. Numerical.** A 4 kg block is released at 35 degrees with mu_s = 0.5 and mu_k = 0.4. Does it slide? Find the friction force and the acceleration. Show a sanity check.
- **C. Graph and simulation interpretation.** On the acceleration graph for mu_s = 0.40 and mu_k = 0.30, what is the acceleration just above the slide angle? Why is the graph not a smooth curve starting at 0?
- **D. Design and optimisation challenge.** Design: choose the ramp angle (whole degrees) so that a crate with mu_s = 0.40 and mu_k = 0.30 slides 3 m in 2 s. Check it starts. How high is the top of a 3 m long ramp? What goes wrong if the surface had mu_k = 0.20 but mu_s = 0.40?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    a = L.accel_from_rest(30, 0.40, 0.30)
    checks.append(("worked example: 30 degrees, mu_k 0.30 gives a = 2.356 m/s^2", abs(a - 2.356) < 0.001))
    checks.append(("worked example: speed 4.71 m/s and distance 4.71 m after 2 s",
                   abs(L.speed_after(a, 2) - 4.712) < 0.01 and abs(L.distance_after(a, 2) - 4.712) < 0.01))
    checks.append(("hand calculation: g (0.5 - 0.3 * 0.8660) = 2.356",
                   abs(9.81 * (0.5 - 0.3 * np.sqrt(3) / 2) - a) < 1e-9))
    checks.append(("slide angle for mu_s 0.40 is 21.80 degrees", abs(L.slide_angle(0.40) - 21.80) < 0.01))
    checks.append(("at 21 degrees the block holds (a = 0), at 22 it slides",
                   L.accel_from_rest(21, 0.4, 0.3) == 0.0 and L.accel_from_rest(22, 0.4, 0.3) > 0))
    checks.append(("mu_s from a measured slip angle of 21.8 degrees is 0.40", abs(L.mu_s_from_angle(21.8) - 0.40) < 0.001))
    th = L.angle_for_acceleration(1.5, 0.30)
    closed = np.degrees(np.arctan(0.30) + np.arcsin(1.5 / (9.81 * np.sqrt(1 + 0.30 ** 2))))
    checks.append(("design angle for a = 1.5 m/s^2 is 25.12 degrees and matches the closed form",
                   abs(th - 25.12) < 0.01 and abs(th - closed) < 1e-6))
    checks.append(("design angle is steep enough to start (tan t > mu_s)", np.tan(np.radians(th)) > 0.40))
    checks.append(("friction drops at slipping: just above the slide angle it is mu_k N, below the static limit",
                   L.friction_acting(10, 22, 0.4, 0.3) < L.friction_acting(10, 21.7, 0.4, 0.3)))
    try:
        L.angle_for_acceleration(20.0, 0.3)
        raised = False
    except ValueError:
        raised = True
    checks.append(("edge case: an acceleration larger than g sin(89 deg) raises ValueError", raised))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
